In [ ]:
import json
import time
import requests
import pycountry
from datetime import datetime, timedelta


Input electricitymaps API key. Only need to run the following cell once.

In [ ]:
electricitymaps_API_key = input("electricitymaps API key: ")
if not electricitymaps_API_key:
    raise ValueError("Please enter electricitymaps API key")

# Get all countries

Country code mapping to match both electricitymap's format with aidatacenterindex's format

In [44]:
manual_country_code_mapping = {
    "Turkey": "TR",
    "Russia": "RU-1",
    "Russia-2": "RU-2",
    "Russia-East": "RU-AS",
}
rename_country_code = {
    # "CL": "CL-SEN",
    "CL-SEN": "CL",
}

rename_country_code_ = {value: key for key, value in rename_country_code.items()}

In [45]:
def get_country_code(country_name, manual_mapping):
    manual_mapped = manual_mapping.get(country_name, None)
    if manual_mapped:
        return manual_mapped
    try:
        # Perform a fuzzy search or lookup by name
        country = pycountry.countries.lookup(country_name)
        code = country.alpha_2
        # code = rename_country_code.get(code, code)
        return code
    except LookupError:
        return "Country not found"

In [46]:
response_all = requests.get("https://aidatacenterindex.com/api/countries.json")
all_countries_json = response_all.json()
all_countries_slug = [item["slug"] for item in response_all.json()["countries"]]
all_countries_name = [item["name"] for item in response_all.json()["countries"]]

In [47]:
all_countries_code = [
    get_country_code(country_name, manual_country_code_mapping) for country_name in all_countries_name
]
countries_slug_name_map = {code: name for code, name in zip(all_countries_code, all_countries_name)}

In [48]:
countries_slug_name_map

{'US': 'United States',
 'IN': 'India',
 'DE': 'Germany',
 'KR': 'South Korea',
 'JP': 'Japan',
 'GB': 'United Kingdom',
 'FR': 'France',
 'AE': 'United Arab Emirates',
 'AU': 'Australia',
 'SG': 'Singapore',
 'SA': 'Saudi Arabia',
 'CA': 'Canada',
 'CN': 'China',
 'ZA': 'South Africa',
 'MY': 'Malaysia',
 'TH': 'Thailand',
 'ES': 'Spain',
 'BR': 'Brazil',
 'IE': 'Ireland',
 'ID': 'Indonesia',
 'IL': 'Israel',
 'TW': 'Taiwan',
 'NO': 'Norway',
 'FI': 'Finland',
 'MX': 'Mexico',
 'NZ': 'New Zealand',
 'SE': 'Sweden',
 'NG': 'Nigeria',
 'NL': 'Netherlands',
 'TR': 'Turkey',
 'IT': 'Italy',
 'VN': 'Vietnam',
 'QA': 'Qatar',
 'EG': 'Egypt',
 'CL': 'Chile',
 'PH': 'Philippines',
 'CO': 'Colombia',
 'DK': 'Denmark',
 'KW': 'Kuwait',
 'RU-1': 'Russia',
 'BH': 'Bahrain',
 'HK': 'Hong Kong',
 'CH': 'Switzerland',
 'MA': 'Morocco',
 'AR': 'Argentina',
 'BE': 'Belgium',
 'AT': 'Austria',
 'PL': 'Poland',
 'PT': 'Portugal',
 'RO': 'Romania',
 'GR': 'Greece',
 'JO': 'Jordan',
 'IS': 'Iceland',
 'PE

Fetch electricitymaps' zone naming system

In [49]:
response_electricity_zones = requests.get("https://api.electricitymaps.com/v3/zones")
electricity_zones_json = response_electricity_zones.json()


def build_electricity_countries(data: dict) -> dict:
    built_data = {}
    for key, value in data.items():
        country_code: str = value["countryCode"]
        current_country: dict = built_data.get(country_code, {})

        # if "countryName" in value:
        if not value.get("zoneParentKey"):
            country_name: str = value["countryName" if "countryName" in value else "zoneName"]
            country_code: str = value["countryCode"]
        else:
            country_name: str = value["zoneName"]
            country_code: str = value["zoneKey"]
        country_name = countries_slug_name_map.get(country_code, country_name)

        if not current_country.get("country_code"):
            current_country["country_code"] = country_code
        if not current_country.get("country_name"):
            current_country["country_name"] = country_name
        if not current_country.get("zone_keys"):
            current_country["zone_keys"] = []
        zone_keys: list[str] = current_country["zone_keys"]
        zone_keys.append({
            "key": country_code,
            "name": country_name,
        })

        built_data[country_code] = current_country
    return built_data


electricity_countries = build_electricity_countries(electricity_zones_json)
electricity_countries_name_map = {value["country_name"]: key for key, value in electricity_countries.items()}

In [50]:
electricity_countries

{'AE': {'country_code': 'AE',
  'country_name': 'United Arab Emirates',
  'zone_keys': [{'key': 'AE', 'name': 'United Arab Emirates'}]},
 'AF': {'country_code': 'AF',
  'country_name': 'Afghanistan',
  'zone_keys': [{'key': 'AF', 'name': 'Afghanistan'}]},
 'AG': {'country_code': 'AG',
  'country_name': 'Antigua and Barbuda',
  'zone_keys': [{'key': 'AG', 'name': 'Antigua and Barbuda'}]},
 'AL': {'country_code': 'AL',
  'country_name': 'Albania',
  'zone_keys': [{'key': 'AL', 'name': 'Albania'}]},
 'AM': {'country_code': 'AM',
  'country_name': 'Armenia',
  'zone_keys': [{'key': 'AM', 'name': 'Armenia'}]},
 'AO': {'country_code': 'AO',
  'country_name': 'Angola',
  'zone_keys': [{'key': 'AO', 'name': 'Angola'}]},
 'AR': {'country_code': 'AR',
  'country_name': 'Argentina',
  'zone_keys': [{'key': 'AR', 'name': 'Argentina'}]},
 'AT': {'country_code': 'AT',
  'country_name': 'Austria',
  'zone_keys': [{'key': 'AT', 'name': 'Austria'}]},
 'AU': {'country_code': 'AU',
  'country_name': 'Aus

In [51]:
for a, b in zip(all_countries_code, all_countries_name):
    print(f"{a:<4}: {b:<20} --- {electricity_countries.get(a, {'country_name': 'N/A'})['country_name']}")

US  : United States        --- United States
IN  : India                --- India
DE  : Germany              --- Germany
KR  : South Korea          --- South Korea
JP  : Japan                --- Japan
GB  : United Kingdom       --- United Kingdom
FR  : France               --- France
AE  : United Arab Emirates --- United Arab Emirates
AU  : Australia            --- Australia
SG  : Singapore            --- Singapore
SA  : Saudi Arabia         --- Saudi Arabia
CA  : Canada               --- Canada
CN  : China                --- China
ZA  : South Africa         --- South Africa
MY  : Malaysia             --- Malaysia
TH  : Thailand             --- Thailand
ES  : Spain                --- Spain
BR  : Brazil               --- Brazil
IE  : Ireland              --- Ireland
ID  : Indonesia            --- Indonesia
IL  : Israel               --- Israel
TW  : Taiwan               --- Taiwan
NO  : Norway               --- Norway
FI  : Finland              --- Finland
MX  : Mexico               ---

# Get datacenters per country
From aidatacenterindex

In [52]:
def get_datacenter_country(country_slug: str) -> dict:
    response_country = requests.get(
        f"https://aidatacenterindex.com/countries/{country_slug}/index.json",
    )
    return response_country.json()

In [ ]:
country_datacenters_json: dict = {}
for country_slug in all_countries_slug:
    country_datacenters_json[country_slug] = get_datacenter_country(country_slug)

    # ? Normalize missing "start_year" by setting it to -inf
    for item in country_datacenters_json[country_slug]["items"]:
        if not item.get("start_year"):
            item["start_year"] = float("-inf")


In [58]:
country_datacenters_json

{'united-states': {'generated_at': '2026-05-23T04:29:53.037Z',
  'license': 'CC BY 4.0',
  'taxonomy': {'type': 'country',
   'name': 'United States',
   'slug': 'united-states',
   'item_count': 84,
   'total_megawatts': 70572},
  'items': [{'id': 'meta-hyperion-ai-data-center-louisiana',
    'title': 'Meta Hyperion AI Data Center (Louisiana)',
    'url': 'https://aidatacenterindex.com/datacenters/meta-hyperion-ai-data-center-louisiana.html',
    'megawatts': 5000,
    'companies': ['Meta', 'Entergy Louisiana'],
    'country': 'United States',
    'region': 'Richland Parish, Louisiana',
    'city': 'Rayville',
    'status': 'under_construction',
    'energy_type': 'On-site gas (initial) + Nuclear PPA (long-term)',
    'ai_focus': 'Training (Titan Cluster for Llama 4+)',
    'start_year': 2024,
    'lat': 32.417,
    'lng': -91.517,
    'sources': ['https://www.wired.com/story/louisiana-hands-meta-a-tax-break-and-power-for-its-biggest-data-center',
     'https://www.opportunitylouisian

In [59]:
country_datacenters_json

{'united-states': {'generated_at': '2026-05-23T04:29:53.037Z',
  'license': 'CC BY 4.0',
  'taxonomy': {'type': 'country',
   'name': 'United States',
   'slug': 'united-states',
   'item_count': 84,
   'total_megawatts': 70572},
  'items': [{'id': 'meta-hyperion-ai-data-center-louisiana',
    'title': 'Meta Hyperion AI Data Center (Louisiana)',
    'url': 'https://aidatacenterindex.com/datacenters/meta-hyperion-ai-data-center-louisiana.html',
    'megawatts': 5000,
    'companies': ['Meta', 'Entergy Louisiana'],
    'country': 'United States',
    'region': 'Richland Parish, Louisiana',
    'city': 'Rayville',
    'status': 'under_construction',
    'energy_type': 'On-site gas (initial) + Nuclear PPA (long-term)',
    'ai_focus': 'Training (Titan Cluster for Llama 4+)',
    'start_year': 2024,
    'lat': 32.417,
    'lng': -91.517,
    'sources': ['https://www.wired.com/story/louisiana-hands-meta-a-tax-break-and-power-for-its-biggest-data-center',
     'https://www.opportunitylouisian

In [60]:
with open("all_datacenter_per_country.json", "w") as f:
    json.dump(country_datacenters_json, fp=f, indent=4)

# Get carbon footprint per country (most recent year)

In [ ]:
last_electricitymaps_access = datetime.now()


def get_carbon_country(country_code: str) -> dict:

    global last_electricitymaps_access
    next_request_time = last_electricitymaps_access + timedelta(seconds=0.2)

    if next_request_time > datetime.now():
        sleep_time = (next_request_time - last_electricitymaps_access).total_seconds()
        time.sleep(sleep_time)

    # print(country_code, country_code in electricity_countries)
    response = requests.get(
        (
            "https://api.electricitymaps.com/v3/carbon-intensity/past-range"
            f"?zone={country_code}&start=2018-06-01T05%3A07%3A00.000Z&end=2025-12-31T05%3A07%3A00.000Z"
            "&disableEstimations=false&temporalGranularity=yearly"
        ),
        headers={"auth-token": f"{electricitymaps_API_key}"},
    )
    last_electricitymaps_access = datetime.now()
    return response.json()

In [ ]:
all_country_electricity = {}
for c in country_datacenters_json.values():
    country_name = c["taxonomy"]["name"]
    mapped_code = electricity_countries_name_map[country_name]
    country = electricity_countries[mapped_code]
    # print(
    #     f"{country_name:>20}: {mapped_code:>15} --> {country['country_code']:<5} "
    #     f"({len(country['zone_keys']):>2} zones)"
    # )
    current_country_electricity = {
        # "zone": "ID",
        # "data": [{
        #     "zone": "ID",
        #     "carbonIntensity": 631,
        #     "datetime": "2018-01-01T00:00:00.000Z",
        #     "updatedAt": "2026-02-08T15:57:16.089Z",
        #     "createdAt": "2025-11-11T11:47:55.841Z",
        #     "emissionFactorType": "lifecycle",
        #     "isEstimated": True,
        #     "estimationMethod": "GENERAL_PURPOSE_ZONE_MODEL"},
        # }, ...],
        "zone": country["country_code"],
        "data": [],  # ? Aggregated carbonIntensity data
        "zone_record": {},  # ? Raw per-zone record data
        "zone_record_by_date": {},  # ? Date-mapped per-zone record data
    }

    # ? Get all zone data
    for zone_ in country["zone_keys"]:
        zone_key: str = zone_["key"]
        zone_name: str = zone_["name"]
        zone_key = rename_country_code_.get(zone_key, zone_key)
        zone_record: dict = get_carbon_country(zone_key)
        current_country_electricity["zone_record"][zone_key] = zone_record

        # ? Flip zone data, use date (year) as key
        zone_record_by_date = {
            (datetime.fromisoformat(record["datetime"]).year): record for record in zone_record["data"]
        }
        # current_country_electricity["zone_record"][zone_key] = zone_record_by_date
        for year, record in zone_record_by_date.items():
            if year not in current_country_electricity["zone_record_by_date"]:
                current_country_electricity["zone_record_by_date"][year] = {}
            current_country_electricity["zone_record_by_date"][year][record["zone"]] = record

    # ? Aggregate all zone data into one country data
    for year, year_record in current_country_electricity["zone_record_by_date"].items():
        year_sum: int = 0
        year_record_count: int = len(year_record)
        for year_zone, zone_record in year_record.items():
            year_sum += zone_record["carbonIntensity"]
        year_avg: float = year_sum / year_record_count
        year_data: dict = {
            "year": year,
            "year_sum": year_sum,
            "year_avg": year_avg,
            "year_record_count": year_record_count,
        }
        current_country_electricity["data"].append(year_data)
    all_country_electricity[country["country_name"]] = current_country_electricity
    # break
# c["taxonomy"]

In [66]:
zone_record

{'zone': 'TZ',
 'carbonIntensity': 388,
 'datetime': '2025-01-01T00:00:00.000Z',
 'updatedAt': '2026-02-06T10:06:57.710Z',
 'createdAt': '2025-11-07T00:03:23.403Z',
 'emissionFactorType': 'lifecycle',
 'isEstimated': True,
 'estimationMethod': 'GENERAL_PURPOSE_ZONE_MODEL'}

In [67]:
all_country_electricity

{'United States': {'zone': 'US',
  'data': [{'year': 2018,
    'year_sum': 27982,
    'year_avg': 466.3666666666667,
    'year_record_count': 60},
   {'year': 2019,
    'year_sum': 27050,
    'year_avg': 450.8333333333333,
    'year_record_count': 60},
   {'year': 2020,
    'year_sum': 25225,
    'year_avg': 420.4166666666667,
    'year_record_count': 60},
   {'year': 2021,
    'year_sum': 25563,
    'year_avg': 426.05,
    'year_record_count': 60},
   {'year': 2022,
    'year_sum': 24984,
    'year_avg': 416.4,
    'year_record_count': 60},
   {'year': 2023,
    'year_sum': 24393,
    'year_avg': 406.55,
    'year_record_count': 60},
   {'year': 2024,
    'year_sum': 23387,
    'year_avg': 389.78333333333336,
    'year_record_count': 60},
   {'year': 2025,
    'year_sum': 23091,
    'year_avg': 384.85,
    'year_record_count': 60}],
  'zone_record': {'US': {'zone': 'US',
    'data': [{'zone': 'US',
      'carbonIntensity': 500,
      'datetime': '2018-01-01T00:00:00.000Z',
      'upda

In [68]:
with open("all_electricity_carbon_intensity_kwh.json", "w") as f:
    json.dump(all_country_electricity, fp=f, indent=4)

# Try connecting all datacenter to the electricity carbon footprint dataset

In [69]:
# country_datacenter_value_ = country_datacenter_value.copy()
country_datacenter_value_

{'generated_at': '2026-05-23T04:29:53.037Z',
 'license': 'CC BY 4.0',
 'taxonomy': {'type': 'country',
  'name': 'United States',
  'slug': 'united-states',
  'item_count': 84,
  'total_megawatts': 70572},
 'items': [{'id': 'meta-hyperion-ai-data-center-louisiana',
   'title': 'Meta Hyperion AI Data Center (Louisiana)',
   'url': 'https://aidatacenterindex.com/datacenters/meta-hyperion-ai-data-center-louisiana.html',
   'megawatts': 5000,
   'companies': ['Meta', 'Entergy Louisiana'],
   'country': 'United States',
   'region': 'Richland Parish, Louisiana',
   'city': 'Rayville',
   'status': 'under_construction',
   'energy_type': 'On-site gas (initial) + Nuclear PPA (long-term)',
   'ai_focus': 'Training (Titan Cluster for Llama 4+)',
   'start_year': 2024,
   'lat': 32.417,
   'lng': -91.517,
   'sources': ['https://www.wired.com/story/louisiana-hands-meta-a-tax-break-and-power-for-its-biggest-data-center',
    'https://www.opportunitylouisiana.gov/news/meta-selects-northeast-louisi

In [75]:
for country_datacenter_key, country_datacenter_value in country_datacenters_json.items():
    # if country_datacenter_key.lower().startswith("unit"):
    #     continue
    taxonomy = country_datacenter_value["taxonomy"]
    name = taxonomy["name"]
    # map_code = electricity_countries_name_map[name]
    # print(electricity_countries_name_map[name], name, "exists" if name in all_country_electricity else "N/A")
    country_electricity = all_country_electricity[name]
    for datacenter in country_datacenter_value["items"]:
        print(f"{name} (since {datacenter['start_year']:>4}): {datacenter['title']}")
        if datacenter["megawatts"]:
            kw = datacenter["megawatts"] * 1000
            hours_per_year = 8760
            grid_carbon_intensity_years = country_electricity["data"]
            for g in grid_carbon_intensity_years:
                if g["year"] == 2025:
                    grid_carbon_intensity = g["year_avg"]
            carbon_impact = kw * hours_per_year * grid_carbon_intensity / 1_000_000
        else:
            carbon_impact = "N/A"
        print(f"  {carbon_impact=}")
    print()
    # break


United States (since 2024): Meta Hyperion AI Data Center (Louisiana)
  carbon_impact=16856430.000000004
United States (since -inf): Stargate AI Supercomputer
  carbon_impact=16856430.000000004
United States (since 2024): Microsoft Project Fairwater (Mount Pleasant, Wisconsin)
  carbon_impact=11125243.8
United States (since 2029): Google Nebraska Mega-Campus (Project Tenaska)
  carbon_impact=9102472.2
United States (since 2024): AWS Project Rainier (New Carlisle, Indiana)
  carbon_impact=8091086.4
United States (since -inf): Diablo Canyon AI Integration Project
  carbon_impact=7605621.216
United States (since 2026): Meta–Vistra Nuclear Uprate Agreement (OH/PA)
  carbon_impact=7079700.6
United States (since -inf): Digital Realty — Project Bunkhouse
  carbon_impact=6169453.38
United States (since 2025): AWS AI Innovation Campuses (Salem and Falls Townships, PA)
  carbon_impact=6068314.8
United States (since 2006): Amazon Web Services — Northern Virginia Hyperscale Campus
  carbon_impact=5